# 파싱된 PDF 데이터로 LLM 질의 테스트

`document_intelligence_test.ipynb`에서 만든 `data/parsed/document_intelligence/<파일명>/content.md`(마크다운 변환 결과)를 컨텍스트로 Azure OpenAI에 던져서 질문/답변을 테스트하는 노트북입니다.

## 사전 준비
1. `document_intelligence_test.ipynb`를 먼저 실행해서 `data/parsed/document_intelligence/` 아래에 문서별 폴더(`raw.json`/`content.md`/`metadata.json`/원본 PDF)가 생성되어 있어야 합니다.
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 Azure OpenAI 값이 채워져 있어야 합니다 (`.env.example` 참고).

```
AZURE_OPENAI_ENDPOINT=https://<your-resource-name>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-api-key>
AZURE_OPENAI_DEPLOYMENT_NAME=<deployment-name>
AZURE_OPENAI_API_VERSION=2024-12-01-preview
```

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AOAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AOAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

PARSED_DIR = Path.cwd().parent / "data" / "parsed" / "document_intelligence"

print("ENDPOINT:", AOAI_ENDPOINT)
print("DEPLOYMENT:", AOAI_DEPLOYMENT)
print("PARSED_DIR:", PARSED_DIR)

## 1. 파싱된 문서 목록 확인

`PARSED_DIR` 아래 폴더(=파일별 결과)를 스캔해서 `content.md`가 있는 것만 사용 가능한 문서로 나열합니다. 여기서 출력되는 `pdf_path`(원본 PDF 파일의 정확한 경로)를 아래 3~5절의 `pdf_path` 인자에 그대로 복사해서 사용하세요.

In [ ]:
def list_parsed_docs() -> list[dict]:
    docs = []
    if not PARSED_DIR.exists():
        return docs
    for folder in sorted(PARSED_DIR.iterdir()):
        md_path = folder / "content.md"
        if not folder.is_dir() or not md_path.exists():
            continue
        meta_path = folder / "metadata.json"
        meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
        pdf_candidates = sorted(folder.glob("*.pdf"))
        docs.append(
            {
                "name": folder.name,
                "filename": meta.get("filename", f"{folder.name}.pdf"),
                "pdf_path": pdf_candidates[0] if pdf_candidates else None,
                "content_md": md_path,
                "raw_json": folder / "raw.json",
                "content_chars": meta.get("content_chars"),
                "pages": meta.get("pages_analyzed"),
            }
        )
    return docs


available_docs = list_parsed_docs()
print(f"사용 가능한 문서 {len(available_docs)}개")
for d in available_docs:
    print(f"- {d['pdf_path']}  (pages={d['pages']}, chars={d['content_chars']})")

## 2. Azure OpenAI 클라이언트 준비

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_API_KEY,
    api_version=AOAI_API_VERSION,
)

## 3. 단일 문서 기준 질문/답변

원본 PDF 파일의 정확한 경로(`pdf_path`, 1절 출력 결과를 그대로 복사)를 지정하면 그 문서의 `content.md` 전체를 컨텍스트로 넣고 질문합니다. 잘라내지 않고 그대로 보내므로, 컨텍스트 길이 제한 에러(400/토큰 초과 등)가 나면 그때 잘라내는 로직을 추가하세요.

In [ ]:
def find_doc(pdf_path) -> dict:
    """원본 PDF 파일의 정확한 경로를 받아 해당 파싱 결과 폴더의 문서를 찾는다.

    document_intelligence_test.ipynb가 만든 폴더(`data/parsed/document_intelligence/<폴더명>/`)에는
    raw.json/content.md/metadata.json과 함께 원본 PDF가 들어있다. 그 PDF 파일의 경로를 그대로 받아서
    부모 폴더를 파싱 결과 폴더로 사용한다 (1절에서 출력되는 pdf_path를 그대로 복사해서 쓰면 된다).
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")

    folder = pdf_path.parent
    md_path = folder / "content.md"
    if not md_path.exists():
        raise FileNotFoundError(
            f"'{folder}'에 content.md가 없습니다. document_intelligence_test.ipynb를 먼저 실행하세요."
        )

    meta_path = folder / "metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    return {
        "name": folder.name,
        "filename": meta.get("filename", pdf_path.name),
        "pdf_path": pdf_path,
        "content_md": md_path,
        "raw_json": folder / "raw.json",
        "content_chars": meta.get("content_chars"),
        "pages": meta.get("pages_analyzed"),
    }


def ask(question: str, pdf_path) -> str:
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")

    system_prompt = (
        "당신은 주어진 문서 내용만 근거로 질문에 답하는 어시스턴트입니다. "
        "문서에 없는 내용은 추측하지 말고 '문서에서 확인할 수 없습니다'라고 답하세요."
    )
    user_prompt = f"[문서: {doc['name']}]\n---\n{content}\n---\n\n질문: {question}"

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content or ""

In [ ]:
# 예시: 표가 깨졌던 파일로 테스트했던 그 문서에 질문해보기 (pdf_path는 1절 출력 결과에서 복사)
answer = ask(
    question="이 약관에서 위치정보 제공 동의 철회는 어떻게 하나요?",
    pdf_path=PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf",
)
print(answer)

## 4. 답변 근거(evidence) 검증 - 원문과 정확히 일치하는지 + 몇 페이지인지 확인

LLM은 답변을 새로 생성하기 때문에 문서 문장을 그대로 베끼지 않고 의역/요약하는 경우가 많습니다. 이를 보완하기 위해:

1. 모델에게 `answer`(자연어 답변)와 `evidence`(근거 문장 — 문서 원문 그대로 인용)를 JSON으로 분리해서 받고,
2. Python 코드로 `evidence`의 각 문장이 실제 `content.md`에 **정확한 substring으로 존재하는지** 검증하고,
3. 발견된 위치(offset)가 `raw.json`의 `pages[].spans` 구간 중 어디에 속하는지로 **몇 페이지인지**도 계산합니다.

즉 "모델의 말"이 아니라 "코드가 원문에서 확인한 것"만 근거로 보여줍니다. ✅는 원문과 100% 일치, 〜는 공백/줄바꿈만 다름(내용은 일치), ❌는 문서에서 찾을 수 없음(모델이 지어냈을 가능성/환각 의심)을 뜻합니다.

Document Intelligence는 전체 텍스트(`content`, 곧 `content.md`와 동일)를 하나의 문자열로 두고, 각 페이지가 그 문자열의 어느 offset 구간([start, end))을 차지하는지를 `pages[].spans`로 기록합니다. evidence 문장이 `content.md`에서 발견된 offset이 어느 페이지 구간에 속하는지 찾으면 페이지 번호를 알 수 있습니다. (`raw.json`이 없으면 페이지 번호는 계산할 수 없습니다.)

In [ ]:
EVIDENCE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "answer_with_evidence",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "answer": {"type": "string"},
                "evidence": {
                    "type": "array",
                    "description": "답변의 근거가 되는 문서 원문 문장 (의역/요약 없이 원문 그대로 복사)",
                    "items": {"type": "string"},
                },
            },
            "required": ["answer", "evidence"],
            "additionalProperties": False,
        },
    },
}


def build_page_spans(doc: dict) -> list[tuple[int, int, int]]:
    """raw.json의 pages[].spans에서 (start_offset, end_offset, page_number) 목록을 만든다."""
    raw_path = doc["raw_json"]
    if not raw_path.exists():
        return []

    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    page_spans = []
    for page in raw.get("pages", []):
        spans = page.get("spans") or []
        if not spans:
            continue
        start = min(s["offset"] for s in spans)
        end = max(s["offset"] + s["length"] for s in spans)
        page_spans.append((start, end, page["pageNumber"]))

    page_spans.sort(key=lambda x: x[0])
    return page_spans


def offset_to_page(offset: int, page_spans: list[tuple[int, int, int]]) -> int | None:
    for start, end, page_number in page_spans:
        if start <= offset < end:
            return page_number
    # 페이지 사이 구분자(<!-- PageBreak --> 등) 위치라 정확히 안 걸치는 경우, 가장 가까운 이전 페이지로 대체
    candidates = [p for p in page_spans if p[0] <= offset]
    return candidates[-1][2] if candidates else None


def find_pages_for_quote(quote: str, content: str, page_spans: list[tuple[int, int, int]]) -> list[int]:
    if not page_spans:
        return []
    pages = set()
    start_idx = 0
    while True:
        idx = content.find(quote, start_idx)
        if idx == -1:
            break
        page = offset_to_page(idx, page_spans)
        if page is not None:
            pages.add(page)
        start_idx = idx + 1
    return sorted(pages)


def ask_with_evidence(question: str, pdf_path) -> dict:
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")
    page_spans = build_page_spans(doc)

    system_prompt = (
        "당신은 주어진 문서 내용만 근거로 질문에 답하는 어시스턴트입니다. "
        "문서에 없는 내용은 추측하지 말고 '문서에서 확인할 수 없습니다'라고 답하세요. "
        "evidence 필드에는 답변의 근거가 되는 문장을 문서 원문 그대로(의역/요약/축약 없이, 띄어쓰기까지 그대로) 복사해서 넣으세요."
    )
    user_prompt = f"[문서: {doc['name']}]\n---\n{content}\n---\n\n질문: {question}"

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=EVIDENCE_SCHEMA,
    )

    parsed = json.loads(response.choices[0].message.content or "{}")

    evidence = []
    for quote in parsed.get("evidence", []):
        exact_match = quote in content
        loosely_match = " ".join(quote.split()) in " ".join(content.split())
        pages = find_pages_for_quote(quote, content, page_spans) if exact_match else []
        evidence.append(
            {
                "quote": quote,
                "exact_match": exact_match,
                "verified": exact_match or loosely_match,
                "pages": pages,
            }
        )

    return {
        "doc": doc["name"],
        "question": question,
        "answer": parsed.get("answer", ""),
        "evidence": evidence,
    }


def print_result(result: dict) -> None:
    print(f"[문서: {result['doc']}]")
    print(f"질문: {result['question']}\n")
    print(f"답변:\n{result['answer']}\n")
    print("근거(evidence):")
    if not result["evidence"]:
        print("  (LLM이 근거 문장을 제시하지 않았습니다)")
    for i, ev in enumerate(result["evidence"], start=1):
        if ev["exact_match"]:
            mark = "✅ 원문과 정확히 일치"
        elif ev["verified"]:
            mark = "〜 내용은 일치 (공백/줄바꿈만 다름)"
        else:
            mark = "❌ 문서에서 찾을 수 없음 (환각 의심)"

        if ev["pages"]:
            page_label = ", ".join(str(p) for p in ev["pages"])
            mark += f" (page {page_label})"
        elif ev["exact_match"]:
            mark += " (페이지 확인 불가: raw.json 없음)"

        print(f"  {i}. {mark}")
        print(f"     \"{ev['quote']}\"")

In [ ]:
result = ask_with_evidence(
    question="이 약관에서 위치정보 제공 동의 철회는 어떻게 하나요?",
    pdf_path=PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf",
)
print_result(result)

## 5. 항목(items) vs 약관 원문 검증

아래 형태의 요청 데이터가 들어오면, 지정한 약관 PDF(들)의 원문과 대조해서 `items`의 각 항목(`itemNm`/`value`)이 그 약관에 실제로 있는 내용인지 판정합니다.

```json
{
  "request_id": "123",
  "objects": [
    {
      "name": "요고 69",
      "items": [
        {"value": "Y", "itemNm": "초이스 상품여부"},
        {"value": "개인, 미성년자, 외국인", "itemNm": "이용 가능 고객"}
      ]
    }
  ]
}
```

- `name`은 상품명일 수도, 다른 대상일 수도 있음 — `items`는 그 `name`에 대한 속성 목록으로 취급되어 약관과 대조됩니다.
- 문서 매칭은 3~4절과 동일하게 `find_doc()`으로 **원본 PDF 파일의 정확한 경로**를 받아 문서를 찾습니다. 경로는 1절에서 출력되는 `pdf_path`를 그대로 복사해서 쓰면 됩니다.
- 지금은 테스트 단계라 `name` → 대조할 PDF 매핑은 자동화하지 않고, `PDF_PATHS`에 **고정된 경로 목록**을 직접 적어서 그 문서들 각각에 대해 검증합니다 (문서별로 결과를 따로 보여줌).
- 판정은 4절의 evidence 검증과 동일하게: 모델이 `evidence`(약관 원문 그대로 인용)를 내면 → 코드가 `content.md`에서 실제 존재하는지/몇 페이지인지 확인 → **"일치"인데 근거가 원문에서 확인되지 않으면** 환각 의심 경고를 표시합니다.
- 판정 값: `일치` / `불일치` / `확인불가`.

In [ ]:
ITEM_VERIFICATION_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "item_verification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "results": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "itemNm": {"type": "string"},
                            "value": {"type": "string"},
                            "evidence": {
                                "type": "array",
                                "description": "먼저 약관 본문에서 이 항목과 관련된 문장을 찾아 원문 그대로(의역/요약 없이) 복사. 관련 내용이 전혀 없으면 빈 배열.",
                                "items": {"type": "string"},
                            },
                            "reason": {
                                "type": "string",
                                "description": "위 evidence를 근거로 이 항목이 value와 맞는지/틀리는지/확인 불가능한지 설명",
                            },
                            "verdict": {
                                "type": "string",
                                "description": "evidence와 reason을 바탕으로 마지막에 결정하는 최종 판정",
                                "enum": ["일치", "불일치", "확인불가"],
                            },
                        },
                        "required": ["itemNm", "value", "evidence", "reason", "verdict"],
                        "additionalProperties": False,
                    },
                },
            },
            "required": ["results"],
            "additionalProperties": False,
        },
    },
}


def verify_items_against_doc(name: str, items: list[dict], pdf_path) -> dict:
    """items(itemNm/value 목록)이 pdf_path(원본 PDF의 정확한 경로)와 일치하는 약관 문서 원문과 일치하는지 검증한다."""
    doc = find_doc(pdf_path)
    content = doc["content_md"].read_text(encoding="utf-8")
    page_spans = build_page_spans(doc)

    items_text = "\n".join(f"- {it['itemNm']}: {it['value']}" for it in items)

    system_prompt = (
        "당신은 항목 정보(itemNm/value)가 실제 약관 원문과 일치하는지 검증하는 어시스턴트입니다. "
        "각 항목마다 반드시 이 순서로 작업하세요: "
        "(1) 먼저 약관 본문에서 이 항목과 관련된 문장을 찾아 evidence에 원문 그대로(의역/요약/축약 없이, 띄어쓰기까지) 복사한다. "
        "관련 내용을 전혀 찾을 수 없으면 evidence는 빈 배열로 둔다. "
        "(2) 그 evidence를 근거로 reason에 이 value가 맞는지/틀리는지/판단 불가능한지 설명한다. "
        "(3) 마지막으로 evidence와 reason에서 실제로 뒷받침된 내용만 바탕으로 verdict를 정한다 — "
        "evidence가 비어있거나 이 항목을 명확히 뒷받침하지 못하면 반드시 '확인불가'로, "
        "evidence가 value와 다른 내용을 명시하면 '불일치'로, "
        "evidence가 value와 일치하는 내용을 명확히 명시할 때만 '일치'로 판정한다. "
        "verdict가 evidence/reason과 모순되면 안 된다."
    )
    user_prompt = (
        f"[대상 명칭: {name}]\n"
        f"[대조할 약관: {doc['name']}]\n---\n{content}\n---\n\n"
        f"다음 항목들이 위 약관 내용과 일치하는지 각각 판정하세요:\n{items_text}"
    )

    response = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=ITEM_VERIFICATION_SCHEMA,
    )
    parsed = json.loads(response.choices[0].message.content or "{}")

    results = []
    for item in parsed.get("results", []):
        evidence = []
        for quote in item.get("evidence", []):
            exact_match = quote in content
            loosely_match = " ".join(quote.split()) in " ".join(content.split())
            pages = find_pages_for_quote(quote, content, page_spans) if exact_match else []
            evidence.append(
                {
                    "quote": quote,
                    "exact_match": exact_match,
                    "verified": exact_match or loosely_match,
                    "pages": pages,
                }
            )
        results.append(
            {
                "itemNm": item.get("itemNm", ""),
                "value": item.get("value", ""),
                "evidence": evidence,
                "reason": item.get("reason", ""),
                "verdict": item.get("verdict", ""),
            }
        )

    return {"name": name, "doc": doc["name"], "results": results}


def verify_request(request: dict, pdf_paths: list) -> list[dict]:
    """request의 각 object(name+items)를 pdf_paths에 적힌 원본 PDF 경로마다 각각 검증한다."""
    all_results = []
    for obj in request.get("objects", []):
        name = obj.get("name", "")
        items = obj.get("items", [])
        for pdf_path in pdf_paths:
            all_results.append(verify_items_against_doc(name, items, pdf_path))
    return all_results


def print_verification(result: dict) -> None:
    print(f"[대상: {result['name']}] vs [약관: {result['doc']}]")
    for item in result["results"]:
        if item["verdict"] == "일치":
            mark = "✅ 일치"
        elif item["verdict"] == "불일치":
            mark = "❌ 불일치"
        else:
            mark = "➖ 확인불가"

        has_verified_evidence = any(e["verified"] for e in item["evidence"])
        if item["verdict"] == "일치" and item["evidence"] and not has_verified_evidence:
            mark += " ⚠️(근거 원문 미확인 - 환각 의심)"

        print(f"  - {item['itemNm']}: {item['value']}  → {mark}")
        print(f"    사유: {item['reason']}")
        for ev in item["evidence"]:
            if ev["exact_match"]:
                tag = "✅ 원문 일치"
            elif ev["verified"]:
                tag = "〜 내용 일치(공백만 차이)"
            else:
                tag = "❌ 원문에서 못 찾음"
            page_label = f" (page {', '.join(str(p) for p in ev['pages'])})" if ev["pages"] else ""
            print(f"      · {tag}{page_label}: \"{ev['quote']}\"")
    print()

### 5-1. 테스트 실행

`PDF_PATHS`에 대조할 약관 원본 PDF의 정확한 경로를 채워 넣고 실행하세요 (1절에서 출력되는 `pdf_path`를 그대로 복사). `objects[].items`는 해당 `objects[].name`에 대한 속성 목록으로 취급되어, 각 PDF마다 한 번씩 검증됩니다.

In [ ]:
REQUEST_SAMPLE = {
    "request_id": "123",
    "objects": [
        {
            "name": "요고 69",
            "items": [
                {"value": "Y", "itemNm": "초이스 상품여부"},
                {"value": "개인, 미성년자, 외국인", "itemNm": "이용 가능 고객"},
            ],
        }
    ],
}

# 검증에 사용할 원본 PDF의 정확한 경로를 채워 넣으세요. (1절에서 출력되는 pdf_path를 그대로 복사)
# 예: PARSED_DIR / "위치기반서비스+이용약관(별표)_20260331_V5.8" / "위치기반서비스+이용약관(별표)_20260331_V5.8.pdf"
PDF_PATHS = [
    # "여기에 원본 PDF의 정확한 경로",
]

results = verify_request(REQUEST_SAMPLE, PDF_PATHS)
for r in results:
    print_verification(r)

### 5-2. request와 같은 구조로 응답 만들기

`print_verification()`은 사람이 눈으로 보기 위한 출력이고, 실제로는 요청받은 JSON과 같은 구조로 `items`마다 `verdict`/`reason`/`evidence`만 채워서 돌려주면 됩니다. `build_response()`가 그 역할을 합니다 (PDF 1개 기준 — 여러 PDF와 대조하고 싶으면 PDF마다 한 번씩 호출).

In [ ]:
def build_response(request: dict, pdf_path) -> dict:
    """request와 동일한 구조(request_id/objects/name/items)를 유지한 채,
    각 item(itemNm/value)에 verdict/reason/evidence만 채워서 반환한다."""
    response_objects = []
    for obj in request.get("objects", []):
        name = obj.get("name", "")
        items = obj.get("items", [])
        verified = verify_items_against_doc(name, items, pdf_path)

        # itemNm+value 조합으로 원본 items 순서와 검증 결과를 매칭
        result_by_key = {(r["itemNm"], r["value"]): r for r in verified["results"]}

        new_items = []
        for it in items:
            key = (it.get("itemNm"), it.get("value"))
            r = result_by_key.get(key, {})
            new_items.append(
                {
                    "itemNm": it.get("itemNm"),
                    "value": it.get("value"),
                    "verdict": r.get("verdict", "확인불가"),
                    "reason": r.get("reason", ""),
                    "evidence": [
                        {"quote": e["quote"], "verified": e["verified"], "pages": e["pages"]}
                        for e in r.get("evidence", [])
                    ],
                }
            )

        response_objects.append({"name": name, "items": new_items})

    return {
        "request_id": request.get("request_id"),
        "doc": verified["doc"] if response_objects else str(pdf_path),
        "objects": response_objects,
    }


# 예시 (5-1의 REQUEST_SAMPLE, PDF_PATHS[0] 사용)
if PDF_PATHS:
    response = build_response(REQUEST_SAMPLE, PDF_PATHS[0])
    print(json.dumps(response, ensure_ascii=False, indent=2))